# Bölüm 4 · Hipotez Testi

[Atölye](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b04/) · [Veri ve lisans](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b04/VERI.md) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b04/GOREVLER.md) · [Raporlama](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b04/RAPORLAMA.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b04/COZUMLER.md)

**Başlangıç:** Google hesabınızla açın, **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU yeterlidir. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk kurulum internet ve birkaç dakika gerektirebilir. Özellikle bootstrap ve yapısal modeller daha uzun sürebilir.

Bu defter, aynı GitHub deposundaki ana Python analizini ve bölümün ileri eklerini çalıştırır. Dosya yüklemeniz veya önce başka bölümü elle yürütmeniz gerekmez. Gerçek veri, öğretim alt kümesi ve simülasyon ayrımı veri rehberinde açıklanır; bunları aynı örneklem türü saymayın. R/SPSS/Amos burada çalıştırılmaz.

## Üç çekirdek kazanım ve ileri rota

1. Hipotezi araştırma hedefi ve tasarımla eşler.
2. Tahmin, güven aralığı ve p değerini birlikte yorumlar.
3. Reddedememe, eşdeğerlik ve nedenselliği ayırır.

**İleri rota:** TOST, çoklu aile ve durdurma simülasyonları; temel aralık/test mantığından sonra.

**Ç1 · Çıkış sorusu:** p=.03, sıfır hipotezinin doğru olma olasılığının yüzde 3 olduğu anlamına gelir mi?

Yanıtınızı çalışma sonunda öğrenci karar günlüğüne yazın. [Rubrik ve teslim rehberi](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b04/RUBRIK.md).

## 1 · Kaynakları hazırla

Sabit bir GitHub sürümündeki öğrenci paketi SHA256 ile doğrulanır. Her çalıştırma ayrı bir geçici çalışma kopyası açar; önceki sonuçlarınız silinmez. Kod ve veri, bu doğrulanan sürümden gelir. Oturum kapanmadan sonuçları indirin.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, urllib.request, zipfile, io, tempfile, subprocess, importlib.metadata
DATA_COMMIT = '23acf4be23823aa139523c8f4db34cff248c285d'
ARCHIVE_SHA256 = 'ce6440f5ee3b4dde9709e1fb250e7b7d8ac86be101b6ababacdab240443560f6'
ARCHIVE_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/ogrenci-paketi-22-bolum.zip"
cache = Path.cwd()/"kitap-colab-kaynak.zip"
if cache.exists():
    raw = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError("Kaynak indirilemedi. İnterneti kontrol edip hücreyi yeniden çalıştırın: "+ARCHIVE_URL) from exc
if hashlib.sha256(raw).hexdigest()!=ARCHIVE_SHA256:
    raise ValueError("Öğrenci paketi SHA256 denetimi başarısız. Kontrolü kaldırmayın; yerel kitap-colab-kaynak.zip dosyasını doğrulayın.")
cache.write_bytes(raw)
work = Path(tempfile.mkdtemp(prefix="b04-atolye-",dir=Path.cwd()))
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(z.infolist())<10000 and sum(i.file_size for i in z.infolist())<134217728
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if work.resolve() not in target.parents or ((member.external_attr>>16)&0o170000)==0o120000:
            raise ValueError("Güvenli olmayan arşiv yolu")
    z.extractall(work)
root = work/"ileri-istatistik-22-bolum"
assert (root/"BOLUM_ESLESMESI.json").is_file()
application=root/'uygulamalar/b04-hipotez-testi'
modules=['uygulamalar/b04-hipotez-testi/esdegerlik-cokluk']
expected_main=json.loads((root/"bolumler/b04/beklenen.json").read_text())
expected_extra={m:json.loads((root/m/"sonuclar/ozet.json").read_text()) for m in modules}
print("Kaynak doğrulandı; Bölüm 4 çalışma kopyası hazır.")

## 2 · Analiz ortamını hazırla

Sınanmış paket sürümleri kullanılır. Mevcut çekirdek bu sürümlerdeyse doğrudan kullanılır. Farklıysa analizler ayrı bir Python ortamında kurulur; Colab çekirdeğindeki paketler topluca değiştirilmez. Kurulum başarısız olursa hücredeki hatayı inceleyin. Python 3.12 veya üzeri çalışma zamanı önerilir. GPU gerekmez.

In [ ]:
FORCE_ISOLATED=globals().get("FORCE_ISOLATED",False)
PINS={'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0', 'matplotlib': '3.11.2', 'statsmodels': '0.15.0', 'PyMuPDF': '1.26.6'}
def package_versions(executable):
    check="import importlib.metadata,json; names="+repr(list(PINS))+";print(json.dumps({n:importlib.metadata.version(n) for n in names}))"
    attempt=subprocess.run([str(executable),"-c",check],capture_output=True,text=True)
    return json.loads(attempt.stdout) if attempt.returncode==0 else {}
python=Path(sys.executable)
if FORCE_ISOLATED or package_versions(python)!=PINS:
    environment=Path.cwd()/".kitap-colab-python"
    if not (environment/"bin/python").exists():
        subprocess.run([sys.executable,"-m","venv",str(environment)],check=True)
    python=environment/"bin/python"
    if package_versions(python)!=PINS:
        subprocess.run([str(python),"-m","pip","install",*[name+"=="+version for name,version in PINS.items()]],check=True)
assert package_versions(python)==PINS,"Paket sürümleri sınanmış ortamla eşleşmiyor"
versions=package_versions(python)
run_env={**os.environ,"MPLBACKEND":"Agg","MPLCONFIGDIR":str(work/"mpl"),"OPENBLAS_NUM_THREADS":"1","OMP_NUM_THREADS":"1","PYTHONDONTWRITEBYTECODE":"1"}
(work/"logs").mkdir(exist_ok=True)
def run_script(relative):
    print("Çalışıyor:",relative,flush=True)
    result=subprocess.run([str(python),str(root/relative)],cwd=root,env=run_env,capture_output=True,text=True,timeout=900)
    (work/"logs"/(relative.replace("/","__")+".log")).write_text(result.stdout+"\n"+result.stderr)
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+"\n"+result.stderr[-4000:])
    print("Tamamlandı.")
print("Analiz ortamı hazır:",versions)

## Ana analiz

Kaynak ve yöntem için yukarıdaki veri rehberini okuyun. Ana analiz ve sonradan eklenen duyarlılık uygulamaları ayrı çalıştırılır. Sonuçlar önceden gösterilen tabloların kopyası olarak değil, kod yeniden yürütülerek üretilir.

In [ ]:
before_figures=set((root/"assets_figures").glob("*.pdf"))
run_script("bolumler/b04/analiz.py")
main_result=json.loads((application/"sonuclar/ozet.json").read_text())
from IPython.display import display, JSON, HTML, Image
print("Ana sonuçlar — ayrıntıları açmak için okları kullanın:")
display(JSON(main_result,expanded=False))

## İleri uygulamalar ve duyarlılık

Her ekin açıklaması atölye sayfasındadır. Bu sonuçları ana analizle veya birbirinden farklı tahmin hedefleriyle karıştırmayın. Hazır kontrol betikleri varsa önce onlar, ardından ek analiz çalışır.

In [ ]:
extra_results={}
for module in modules:
    if (root/module/"dogrula.py").is_file(): run_script(module+"/dogrula.py")
    run_script(module+"/analiz.py")
    extra_results[module]=json.loads((root/module/"sonuclar/ozet.json").read_text())
    print(module)
    display(JSON(extra_results[module],expanded=False))

## Sonuçları referanslarla karşılaştır

Kitabın sayısal referansları, önceki sürümden değişmeden alınır. Sürüm/zaman gibi hesap dışı alanlar mevcut karşılaştırma politikasına göre dışlanır. Hata oluşursa veri, sürüm ve günlük dosyalarını kontrol edin; referansları sonuca uydurmayın. Bu kontrol bilimsel varsayımları veya SPSS yürütmesini doğrulamaz.

In [ ]:
sys.path.insert(0,str(root/"tools"))
from dogrula_22 import compare
checks={"main":compare(expected_main,main_result)}
for module in modules:
    checks[module]=compare(expected_extra[module],json.loads((root/module/"sonuclar/ozet.json").read_text()))
assert all(not differences for differences in checks.values()),checks
print("Ana analiz ve",len(modules),"ileri ek için sayısal referanslar eşleşti.")

## Grafikler ve özet tablolar

Grafikler bu çalıştırmada üretilir. Ön koşul olarak çalıştırılan Bölüm 18'in şekilleri burada tekrar gösterilmez. CSV tablolarının ilk 12 satırı görüntülenir; tam dosyalar çıktı paketindedir. Şekil sayısı bölüme göre değişir.

In [ ]:
import csv,html
new_figures=sorted(set((root/"assets_figures").glob("*.pdf"))-before_figures)
for figure in new_figures:
    png=figure.with_suffix(".png")
    renderer="import fitz,sys;d=fitz.open(sys.argv[1]);d[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(sys.argv[2])"
    subprocess.run([str(python),"-c",renderer,str(figure),str(png)],check=True)
    print(figure.name)
    display(Image(filename=str(png)))
for table in sorted((application/"sonuclar").glob("*.csv"))[:3]:
    with table.open(encoding="utf-8-sig",newline="") as f: rows=list(csv.reader(f))
    content="<h4>"+html.escape(table.name)+"</h4><table>"
    for i,row in enumerate(rows[:13]):
        tag="th" if i==0 else "td"
        content+="<tr>"+"".join("<"+tag+">"+html.escape(value)+"</"+tag+">" for value in row)+"</tr>"
    display(HTML(content+"</table>"))

## Sıra sizde

# Bölüm 4 — görevler

Güncel kitap bölümünün etkinlik ve alıştırmalarıdır. Önce kendi çözümünüzü
üretin; ardından çözümleri kontrol edin. Kitap sayfa numaraları PDF sürümüne
göre değişebilir.

## D1–D3: Testten Önce

1.  Neden $n=20$ değil $n=10$ kullanılır? Farkın yönü nasıl tanımlanır?

2.  Koşul adları ve saat birimi hangi bilgiyi sağlar; hangi tasarım bilgilerini sağlamaz?

3.  Ana analizden önce yön, alfa ve dışlama kararı neden yazılmalıdır? Tarihsel veride öğretim tercihi gerçek önkayıt mıdır?

# Rehberli ve Bağımsız Etkinlikler

## G1–G4: Rehberli Çözüm

1.  Gerçek farklardan n, ortalama, s, SH, t ve serbestlik derecesini yeniden hesaplayın.

2.  Aynı modelle yüzde 95 aralığı bulun; sıfırın dışlanması ile p kararını eşleştirin.

3.  Dolum örneğinin iki yönlü p’sini ve aralığını yorumlayın. Azalışa sonradan geçmek niçin aynı plan değildir?

4.  Sıfır simülasyonunda Tip I hata ve güven aralığı kapsamasını bulun; .5 senaryosunda hangi oran güçtür?

## B1–B3: Bağımsız Çözüm

1.  Uyku farkını koşul 1 eksi koşul 2 yaparsanız t, iki yönlü p, sağ p ve aralık nasıl değişir?

2.  Kurgusal çapraz tablodaki beklenen sayıları, ki-kareyi ve V’yi hesaplayın. Yöntem seçimi niçin belirtilmelidir?

3.  Bağımsız 20 doğru sıfır hipotezinde aile hatasını ve Bonferroni tekil eşiğini bulun. Sonuç bağımlı testlere aynen taşınabilir mi?

## H1: Hatalı Raporu Onarma

> “p=.0028, sıfır hipotezinin doğru olma olasılığı yüzde .28. Yirmi satırımız var, n=20. Dolumda p=.057 çıktı; tam 500 ml kanıtlandı. Yönü değiştirip .028 elde ettik. Uyku farkının aralığı bütün hastaların yüzde 95’ini kapsar. Simülasyon gücü .67 olduğundan bu gerçek çalışmanın gücü de .67.”

Altı iddiayı parametre, eşleşme, yön, aralık ve simülasyon ayrımıyla düzeltin.

## P1: Önceden Yazılmış Test Planı

Yeni bir araştırma için hedef parametre, örnekleme/atama birimi, ölçüm zamanı, birincil karşılaştırma, yön/alfa, planlanan n/durdurma, eksikler, varsayımlar, çoklu aile, etki/aralık ve raporlama kararlarını yazın. Gerçekte yapılmamış önkayıt, veri toplama veya güç analizini yapılmış gibi sunmayın. Rubrik: soru/tasarım 25, hipotez/yön 20, hesap/varsayım 25, belirsizlik/çokluk 20, yeniden üretim 10 puan.

Yanıtlar Ek F, Bölüm 4’tedir (s. kitaptaki ilgili sayfa).

# Alıştırmalar

Bu alıştırmalar, hipotez kurma, test istatistiği hesaplama, $p$ değeri yorumlama, hata türleri, güç ve test seçimi becerilerini pekiştirmek için üç düzeyde düzenlenmiştir. Yanıtlar ve kısa çözüm ipuçları Ek F’dedir.

## Kolay Düzey

1.  Tip I hata hangi durumda ortaya çıkar?

2.  İki yönlü bir ortalama testinde alternatif hipotez genel olarak nasıl yazılır?

3.  Test gücü hangi ifade ile gösterilir ve neyi temsil eder?

4.  Bağımsız iki örneklem t testinde varyans homojenliği hangi testle değerlendirilir?

5.  Ki-kare bağımsızlık testi hangi tür değişkenler için uygundur?

## Orta Düzey

1.  Bir üretim hattında ortalama dolum miktarının 500 ml olduğu iddia edilmektedir. $n=25$, $\bar{x}=496$, $s=10$ için iki yönlü hipotezleri kurunuz ve test istatistiğini hesaplayınız.

2.  Bir okulda yeni yöntemin başarı ortalamasını 70’in üzerine çıkarıp çıkarmadığı araştırılıyor. Uygun $H_0$ ve $H_1$ hipotezlerini yazınız.

3.  İki farklı öğretim yönteminin başarı puanları farklı öğrenciler üzerinde karşılaştırılıyor. Hangi test tercih edilir ve hangi varsayımlar kontrol edilir?

4.  İki yönlü bir testte %95 güven aralığı sıfır hipotezindeki referans değeri içermiyorsa karar ne olur?

5.  Bir testte $p=.032$ ve $\alpha=.05$ bulunmuştur. İstatistiksel karar ve raporlama cümlesi nasıl kurulmalıdır?

## İleri Düzey

1.  $n=36$, $\bar{x}=52$, $s=12$ ve $H_0:\mu=50$ için tek örneklem t istatistiğini hesaplayınız. Sonucu yorumlamak için hangi ek bilgi gerekir?

2.  Bir araştırmada $p=.08$ bulunmuştur. Araştırmacı “etki yoktur” sonucunu yazmak istiyor. Bu yorum neden sorunludur?

3.  Büyük örneklemli bir çalışmada çok küçük bir fark için $p<.001$ bulunmuştur. Etki büyüklüğü ve pratik önem neden ayrıca raporlanmalıdır?

4.  Ki-kare bağımsızlık testinde bazı beklenen frekansların 5’in altında olması ne tür bir sorun doğurur? Olası çözüm yollarını belirtiniz.

5.  Tip II hata olasılığı, örneklem büyüklüğü, etki büyüklüğü ve ölçüm hatası arasındaki ilişkiyi araştırma planlama açısından açıklayınız.


## Revizyon 8: eşdeğerlik ve çokluk

## T1–T3: Kararı Hedef ve Hata Güvencesiyle Eşleştirme

1.  Uyku verisinin %90 aralığı $[.866995,2.293005]$ için öğretim sınırları $\Delta=1$ ve 3 altında TOST kararlarını verin. Sıfıra karşı p=.002833 ile bu kararlar neden çelişmez? Klinik eşdeğerlik denebilir mi?

2.  $\bar D=0$, $s=10$, $\Delta=5$ ve $n=25/4$ kurgularında iki farklı TOST sonucunu yorumlayın. İki yönlü p=1 neden tek başına “etki yok” anlamına gelmez?

3.  Ham p ailesi $(.20,.004,.024,.012)$ için sıralamayı, Holm eşiklerini ve özgün sırada düzeltilmiş p’leri hesaplayın. Bu p’lerle düzeltmesiz %95 aralıkları aynı aile güvencesine sahipmiş gibi raporlayabilir miyiz?


Önce kendi yorumunuzu yazın; sonra [çözümlerle](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b04/COZUMLER.md) karşılaştırın. Sonuç, belirsizlik, varsayım ve sınırlılıkları [raporlama rehberine](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/pedagojik-22/bolumler/b04/RAPORLAMA.md) göre ayırın.

In [ ]:
yanitlarim={"bulgular":"", "belirsizlik":"", "varsayimlar":"", "sinirliliklar":""}
# Metinleri doldurun; sayısal kontrolü geçmek bu yorumları tamamlamaz.

## Çıktıları indir

Son hücre sonuç JSON/CSV'lerini, grafiklerin PDF/PNG dosyalarını, günlükleri ve kendi yanıtlarınızı paketler. Colab'ın solundaki Dosyalar panelini yenileyin ve aşağıda yazan ZIP'i indirin. Geçici çalışma zamanı kapanmadan bunu yapın; kod Drive'a otomatik erişmez.

## Öğrenci kararı ve kısa savunma

**Karar görevi:** Tek veya iki yönlü hipotezi sonuçlara bakmadan gerekçelendirin; aynı tahminin yorumunun ve önceden seçilen test planının nasıl değiştiğini gösterin.

Yalnız bir kararı değiştirin. Ana sonucu koruyun; gerekirse yeni bir hesap hücresi ekleyip denemeyi ayrı kaydedin. Sayısal referansları yeni sonuca uydurmayın. Önce/sonra kanıtını sonuç dosyası, tablo veya açık bir hesapla ilişkilendirin. Yapılmamış hesabı yapılmış gibi yazmayın.

**3–5 dakikalık savunma:** Hangi hedefi seçtiniz ve neden? Tam olarak ne değişti? Hangi belirsizlik ve kapsam sınırı kaldı? Karşılaştırma aynı hipotezi mi sınadı?

Aşağıdaki alanları kendi çalışmanızla doldurun. **Tümünü çalıştır**, bu alanları sizin yerinize tamamlamaz. Sayısal kontrollerin geçmesi öğrenci kararının tamamlandığı anlamına gelmez. Alanların dolu olması da kanıtın doğruluğunu otomatik onaylamaz.

In [ ]:
karar_gunlugu = {
    "bolum": 4,
    "gorev": 'Tek veya iki yönlü hipotezi sonuçlara bakmadan gerekçelendirin; aynı tahminin yorumunun ve önceden seçilen test planının nasıl değiştiğini gösterin.',
    "kararim": "",
    "gerekcem": "",
    "degistirdigim_tek_sey": "",
    "onceki_kanit": "",
    "sonraki_kanit": "",
    "belirsizlik_ve_sinir": "",
    "kisa_savunma": "",
    "cikis_yanitim": ""
}
zorunlu_alanlar = ["kararim","gerekcem","degistirdigim_tek_sey","onceki_kanit","sonraki_kanit","belirsizlik_ve_sinir","kisa_savunma","cikis_yanitim"]
karar_gunlugu["alanlar_dolu"] = all(isinstance(karar_gunlugu[k],str) and bool(karar_gunlugu[k].strip()) for k in zorunlu_alanlar)
karar_gunlugu["durum"] = "Öğretim elemanı incelemesi gerekir" if karar_gunlugu["alanlar_dolu"] else "Öğrenci kararı henüz tamamlanmadı"
print(karar_gunlugu["durum"])
# Bu hücreyi doldurduktan sonra çıktı ZIP'i hücresini yeniden çalıştırın.

In [ ]:
manifest={"chapter":4,"source_commit":DATA_COMMIT,"source_archive_sha256":ARCHIVE_SHA256,"versions":versions,"checks":checks,"answers":yanitlarim,"scope":"Python ana analiz ve ileri ekler; SPSS/R/Amos çalıştırılmadı"}
# OGRETIM_EXPORT_BEGIN
manifest["ogrenci_karari"] = karar_gunlugu
(work/"ogrenci-karari.json").write_text(json.dumps(karar_gunlugu,ensure_ascii=False,indent=2),encoding="utf-8")
# OGRETIM_EXPORT_END
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive=Path.cwd()/"b04-colab-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(work/"ogrenci-karari.json","ogrenci-karari.json")
    z.write(work/"calisma-kaydi.json","calisma-kaydi.json")
    for module in ['uygulamalar/b04-hipotez-testi',*modules]:
        for f in sorted((root/module/"sonuclar").rglob("*")):
            if f.is_file() and f.suffix in [".json",".csv"]:z.write(f,str(f.relative_to(root)))
    for figure in new_figures:
        for f in [figure,figure.with_suffix(".png")]: z.write(f,"grafikler/"+f.name)
    for f in (work/"logs").glob("*.log"):z.write(f,"logs/"+f.name)
print("İndirmeye hazır:",archive.name)